In [ ]:
!sudo apt-get install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 0 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (889 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122809 files and director

In [ ]:
!curl -sL https://database.lichess.org/standard/lichess_db_standard_rated_2026-08.pgn.zst | zstd -d | head -n 100000 > august_partial.pgn

In [ ]:
with open('august_partial.pgn', 'r', encoding='utf-8') as f:
    text = f.read()

In [ ]:
len(text)

11869511

In [ ]:
print(text[:1000])

[Event "Rated Bullet tournament https://lichess.org/tournament/summer26"]
[Site "https://lichess.org/OGYhDXDv"]
[Date "2026.08.01"]
[Round "-"]
[White "Babak-Chalus"]
[Black "DAN-NC"]
[Result "0-1"]
[UTCDate "2026.08.01"]
[UTCTime "00:00:01"]
[WhiteElo "1729"]
[BlackElo "1735"]
[WhiteRatingDiff "-5"]
[BlackRatingDiff "+5"]
[ECO "A00"]
[Opening "Saragossa Opening"]
[TimeControl "60+0"]
[Termination "Time forfeit"]

1. c3 { [%clk 0:01:00] } 1... d5 { [%clk 0:01:00] } 2. d3 { [%clk 0:01:00] } 2... c6 { [%clk 0:00:57] } 3. Nd2 { [%clk 0:01:00] } 3... Nf6 { [%clk 0:00:57] } 4. Qc2 { [%clk 0:01:00] } 4... Bf5 { [%clk 0:00:55] } 5. e4 { [%clk 0:01:00] } 5... dxe4 { [%clk 0:00:54] } 6. dxe4 { [%clk 0:01:00] } 6... Bg6 { [%clk 0:00:53] } 7. Ngf3 { [%clk 0:01:00] } 7... e6 { [%clk 0:00:52] } 8. Be2 { [%clk 0:01:00] } 8... Bd6 { [%clk 0:00:51] } 9. a4 { [%clk 0:01:00] } 9... O-O { [%clk 0:00:50] } 10. O-O { [%clk 0:01:00] } 10... Nbd7 { [%clk 0:00:49] } 11. Re1 { [%clk 0:00:58] } 11... Qc7 { [%cl

In [28]:
# first 18 lines are the game information like event name, players and ratings, etc
# then there are moves
# the moves have a game metadata for the time the player has to respond for the move played
# we need to only get the core moves and strip out the time as well as remove all the game info for each game and make it as a beginning of a game and end of a game.
# the beginning will be any move with pawn on allowed coordinate and ending will be a checkmate or time or draw etc - total 8 ways # we will not explicitly put this int the moves the statistical relationship must figure this out.
# the game ends with a score as 0-1 or 1-0 # we need this so that the model can optimize for winning moves and attend to the moves of winners # hypothesis on attention.
# so we need to extract out the moves from '1.' (start of game) till '0-1' or '1-0' (end of game)
# i guess we need the time part as well so the model can learn the time part in win and loss
print(text[:1000].splitlines()[18:])

['1. c3 { [%clk 0:01:00] } 1... d5 { [%clk 0:01:00] } 2. d3 { [%clk 0:01:00] } 2... c6 { [%clk 0:00:57] } 3. Nd2 { [%clk 0:01:00] } 3... Nf6 { [%clk 0:00:57] } 4. Qc2 { [%clk 0:01:00] } 4... Bf5 { [%clk 0:00:55] } 5. e4 { [%clk 0:01:00] } 5... dxe4 { [%clk 0:00:54] } 6. dxe4 { [%clk 0:01:00] } 6... Bg6 { [%clk 0:00:53] } 7. Ngf3 { [%clk 0:01:00] } 7... e6 { [%clk 0:00:52] } 8. Be2 { [%clk 0:01:00] } 8... Bd6 { [%clk 0:00:51] } 9. a4 { [%clk 0:01:00] } 9... O-O { [%clk 0:00:50] } 10. O-O { [%clk 0:01:00] } 10... Nbd7 { [%clk 0:00:49] } 11. Re1 { [%clk 0:00:58] } 11... Qc7 { [%cl']


In [ ]:
# Moves extraction code, concatenated <start> and <end>


In [ ]:
# compile the dataset into train, test and eval (X, Y) pairs # X is input for nn and Y is the respective label or output to be verified nn on.
# Data cleanup & filtering
# We need to take all the moves ?
# Do we need the white and black distinction - no not really # as the model can definitely learn the white to move first as the rule - which we do not want it to build.
# How to get just the moves ?